In [ ]:
%%time
from openai import OpenAI
import pandas as pd
import time 

client = OpenAI(
  base_url = "https://openrouter.ai/api/v1",
  #api_key = " ",
  api_key = " " #my-own-api-key
  
)

CPU times: total: 1.62 s
Wall time: 1.99 s


In [2]:
tools = [
    {
      "type": "function",
      "function": {
        "name": "get_employee_profile",
        "description": "Retrieve complete employee profile information including personal details, employment history, and performance metrics.",
        "parameters": {
          "type": "object",
          "properties": {
            "employee_id": {
              "type": "string",
              "description": "Unique identifier for the employee"
            }
          },
          "required": [
            "employee_id"
          ]
        }        
      }
    },
    {
      "type": "function",
      "function": {
        "name": "schedule_performance_review",
        "description": "Schedule a performance review meeting.",
        "parameters": {
          "type": "object",
          "properties": {
            "employee_id": {
              "type": "string",
              "description": "Employee ID"
            },
            "manager_id": {
              "type": "string",
              "description": "Manager ID"
            },
            "review_type": {
              "type": "string",
              "enum": [
                "quarterly",
                "annual",
                "probationary",
                "promotion"
              ],
              "description": "Review type"
            },
            "scheduled_date": {
              "type": "string",
              "description": "Review date (ISO format)"
            },
            "duration_minutes": {
              "type": "integer",
              "description": "Meeting duration (mins)"
            }
          },
          "required": [
            "employee_id",
            "manager_id",
            "review_type",
            "scheduled_date"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "generate_payroll_report",
        "description": "Generate payroll report for selected employees and period.",
        "parameters": {
          "type": "object",
          "properties": {
            "employee_ids": {
              "type": "array",
              "items": {
                "type": "string"
              },
              "description": "List of employee IDs to include in the report"
            },
            "start_date": {
              "type": "string",
              "format": "date",
              "description": "Start date (YYYY-MM-DD)"
            },
            "end_date": {
              "type": "string",
              "format": "date",
              "description": "End date (YYYY-MM-DD)"
            },
            "pay_frequency": {
              "type": "string",
              "enum": [
                "weekly",
                "bi-weekly",
                "monthly",
                "semi-monthly"
              ],
              "description": "Pay frequency"
            },
            "output_format": {
              "type": "string",
              "enum": [
                "pdf",
                "excel",
                "csv",
                "json"
              ],
              "description": "Report format"
            },
            "country": {
              "type": "string",
              "description": "Country for tax compliance (e.g., 'US')"
            }
          },
          "required": [
            "employee_ids",
            "start_date",
            "end_date",
            "output_format",
            "country"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "get_customer_info",
        "description": "Retrieve basic customer information and contact details.",
        "parameters": {
          "type": "object",
          "properties": {
            "customer_id": {
              "type": "string",
              "description": "Unique customer identifier"
            }
          },
          "required": [
            "customer_id"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "create_sales_opportunity",
        "description": "Create a new sales opportunity with lead information and initial assessment.",
        "parameters": {
          "type": "object",
          "properties": {
            "customer_id": {
              "type": "string",
              "description": "Associated customer ID"
            },
            "opportunity_name": {
              "type": "string",
              "description": "Name/title of the sales opportunity"
            },
            "estimated_value": {
              "type": "number",
              "description": "Estimated deal value in USD"
            },
            "probability": {
              "type": "number",
              "minimum": 0,
              "maximum": 100,
              "description": "Probability of closing the deal (0-100%)"
            },
            "expected_close_date": {
              "type": "string",
              "format": "date",
              "description": "Expected date to close the deal"
            }
          },
          "required": [
            "customer_id",
            "opportunity_name",
            "estimated_value",
            "probability"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "track_customer_interaction",
        "description": "Log customer interactions.",
        "parameters": {
          "type": "object",
          "properties": {
            "customer_id": {
              "type": "string",
              "description": "Customer ID"
            },
            "interaction_type": {
              "type": "string",
              "enum": [
                "phone_call",
                "email",
                "chat",
                "support_ticket"
              ],
              "description": "Type of interaction"
            },
            "channel": {
              "type": "string",
              "enum": [
                "phone",
                "email",
                "live_chat",
                "website"
              ],
              "description": "Channel used"
            },
            "summary": {
              "type": "string",
              "description": "Brief summary"
            },
            "outcome": {
              "type": "string",
              "enum": [
                "positive",
                "neutral",
                "negative"
              ],
              "description": "Interaction result"
            },
            "employee_id": {
              "type": "string",
              "description": "Primary employee involved"
            },
            "follow_up_required": {
              "type": "boolean",
              "description": "Needs follow-up?"
            }
          },
          "required": [
            "customer_id",
            "interaction_type",
            "summary",
            "outcome",
            "employee_id"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "check_stock_level",
        "description": "Check current stock level for a specific product.",
        "parameters": {
          "type": "object",
          "properties": {
            "product_id": {
              "type": "string",
              "description": "Product identifier to check stock for"
            }
          },
          "required": [
            "product_id"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "update_inventory_count",
        "description": "Update product inventory quantity",
        "parameters": {
          "type": "object",
          "properties": {
            "product_id": {
              "type": "string",
              "description": "Product identifier"
            },
            "quantity_change": {
              "type": "integer",
              "description": "Quantity delta (positive/negative)"
            },
            "reason": {
              "type": "string",
              "enum": [
                "sale",
                "purchase",
                "return",
                "damage",
                "adjustment"
              ],
              "description": "Change reason"
            },
            "updated_by": {
              "type": "string",
              "description": "Employee ID performing update"
            }
          },
          "required": [
            "product_id",
            "quantity_change",
            "reason",
            "updated_by"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "generate_inventory_forecast",
        "description": "Generate inventory demand forecast.",
        "parameters": {
          "type": "object",
          "properties": {
            "forecast_period": {
              "type": "integer",
              "description": "Days to forecast ahead"
            },
            "product_categories": {
              "type": "array",
              "description": "Categories to forecast",
              "items": {
                "type": "string"
              }
            },
            "historical_period_days": {
              "type": "integer",
              "description": "Historical data days"
            },
            "seasonality_adjustment": {
              "type": "boolean",
              "description": "Apply seasonal adjustments"
            },
            "confidence_level": {
              "type": "number",
              "description": "Forecast confidence (0.8-0.99)"
            }
          },
          "required": [
            "forecast_period",
            "product_categories"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "process_invoice_payment",
        "description": "Process an invoice payment.",
      "parameters": {
        "type": "object",
        "properties": {
          "invoice_id": {
            "type": "string",
            "description": "Invoice ID."
          },
          "vendor_id": {
            "type": "string",
            "description": "Vendor ID."
          },
          "amount": {
            "type": "number",
            "description": "Payment amount."
          },
          "currency": {
            "type": "string",
            "enum": ["USD", "EUR", "GBP"],
            "default": "USD"
          },
          "payment_method": {
            "type": "string",
            "enum": ["ACH", "Wire", "Check"],
            "default": "ACH"
          }
        },
        "required": ["invoice_id", "vendor_id", "amount"]
      }
    }
  },
    {
      "type": "function",
      "function": {
        "name": "handle_expense_reimbursement",
        "description": "Submit employee expenses for reimbursement.",
        "parameters": {
          "type": "object",
          "properties": {
            "employee_id": {
              "type": "string",
              "description": "Employee ID."
            },
            "expenses": {
              "type": "array",
              "items": {
                "type": "object",
                "properties": {
                  "category": {
                    "type": "string",
                    "enum": [
                      "Travel",
                      "Meals",
                      "Office Supplies"
                    ]
                  },
                  "amount": {
                    "type": "number"
                  },
                  "receipt_attached": {
                    "type": "boolean"
                  }
                },
                "required": [
                  "category",
                  "amount"
                ]
              }
            },
            "approver_id": {
              "type": "string",
              "description": "Approver ID."
            }
          },
          "required": [
            "employee_id",
            "expenses",
            "approver_id"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "execute_month_end_close",
        "description": "Initiate month-end financial close.",
        "parameters": {
          "type": "object",
          "properties": {
            "period": {
              "type": "string",
              "description": "Closing period (YYYY-MM)."
            },
            "tasks": {
              "type": "array",
              "description": "Tasks to execute (e.g., accruals, reconciliations).",
              "items": {
                "type": "string",
                "enum": [
                  "accruals",
                  "depreciation",
                  "bank_reconciliation",
                  "revenue_recognition",
                  "inventory_valuation"
                ]
              }
            },
            "auto_reconcile": {
              "type": "boolean",
              "description": "Auto-match transactions.",
              "default": True
            },
            "reports": {
              "type": "array",
              "description": "Reports to generate.",
              "items": {
                "type": "string",
                "enum": [
                  "P&L",
                  "Balance Sheet",
                  "Cash Flow",
                  "Variance Analysis"
                ]
              }
            }
          },
          "required": [
            "period",
            "tasks"
        ]
      }
    }
    },
    {
      "type": "function",
      "function": {
        "name": "create_helpdesk_ticket",
        "description": "Create an IT support ticket.",
        "parameters": {
          "type": "object",
          "properties": {
            "user_id": {
              "type": "string",
              "description": "Employee ID"
            },
            "issue_summary": {
              "type": "string",
              "description": "Brief issue summary"
            },
            "category": {
              "type": "string",
              "enum": [
                "hardware",
                "software",
                "network",
                "security",
                "email",
                "phone",
                "printer",
                "access"
              ],
              "description": "Issue category"
            },
            "priority": {
              "type": "string",
              "enum": [
                "low",
                "medium",
                "high",
                "critical"
              ],
              "description": "Priority level"
            },
            "description": {
              "type": "string",
              "description": "Detailed problem description"
            },
            "attachments": {
              "type": "array",
              "items": {
                "type": "object",
                "properties": {
                  "filename": {
                    "type": "string",
                    "description": "Attachment filename"
                  },
                  "file_type": {
                    "type": "string",
                    "description": "File type (e.g., image, log)"
                  }
                }
              },
              "description": "Attached files"
            }
          },
          "required": [
            "user_id",
            "issue_summary",
            "category",
            "description"         
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "initiate_remote_session",
        "description": "Start a secure remote desktop session to troubleshoot an employee's device.",
        "parameters": {
          "type": "object",
          "properties": {
            "technician_id": {
              "type": "string",
              "description": "ID of the IT technician initiating the session."
            },
            "employee_device_id": {
              "type": "string",
              "description": "Device ID of the employee needing assistance."
            },
            "permission_flags": {
              "type": "object",
              "properties": {
                "file_transfer": {
                  "type": "boolean",
                  "default": False
                },
                "full_control": {
                  "type": "boolean",
                  "default": False
                },
                "session_recording": {
                  "type": "boolean",
                  "default": True
                }
              },
              "required": [
                "file_transfer",
                "full_control",
                "session_recording"
              ]
            },
            "session_timeout_minutes": {
              "type": "integer",
              "description": "Max duration of the remote session."
            },
            "notify_employee": {
              "type": "boolean",
              "description": "Send a notification to the employee before connecting.",
              "default": True
            }
          },
          "required": [
            "technician_id",
            "employee_device_id",
            "session_timeout_minutes"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "request_network_access",
        "description": "Grant/revoke network access (VPN, Wi-Fi, firewall rules) for an employee.",
        "parameters": {
          "type": "object",
          "properties": {
            "employee_id": {
              "type": "string"
            },
            "access_type": {
              "type": "string",
              "enum": [
                "VPN",
                "Wi-Fi",
                "Firewall"
              ],
              "description": "Type of network access needed."
            },
            "duration_hours": {
              "type": "integer",
              "description": "Duration of temporary access."
            },
            "justification": {
              "type": "string"
            },
            "approver_id": {
              "type": "string",
              "description": "ID of the manager approving the request."
            },
            "ports": {
              "type": "array",
              "items": {
                "type": "integer"
              },
              "description": "Specific ports to open (for firewall)."
            }
          },
          "required": [
            "employee_id",
            "access_type",
            "duration_hours",
            "justification",
            "approver_id"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "create_purchase_order",
        "description": "Generate a purchase order for a vendor.",
        "parameters": {
          "type": "object",
          "properties": {
            "vendor_id": {
              "type": "string",
              "description": "Vendor ID."
            },
            "items": {
              "type": "array",
              "items": {
                "type": "object",
                "properties": {
                  "product_id": {
                    "type": "string"
                  },
                  "quantity": {
                    "type": "integer"
                  },
                  "unit_price": {
                    "type": "number"
                  },
                  "tax_rate": {
                    "type": "number"
                  },
                  "discount": {
                    "type": "number"
                  }
                },
                "required": [
                  "product_id",
                  "quantity",
                  "unit_price"
                ]
              }
            },
            "delivery_date": {
              "type": "string",
              "description": "Delivery date (YYYY-MM-DD)."
            },
            "payment_terms": {
              "type": "string",
              "enum": [
                "Net 30",
                "Net 60",
                "Due on Receipt"
              ]
            },
            "shipping_address": {
              "type": "object",
              "properties": {
                "street": {
                  "type": "string"
                },
                "city": {
                  "type": "string"
                },
                "country": {
                  "type": "string"
                }
              },
              "required": [
                "street",
                "city",
                "country"
              ]
            }
          },
          "required": [
            "vendor_id",
            "items",
            "delivery_date",
            "shipping_address"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "request_vendor_quotation",
        "description": "Request price quotations from multiple vendors for a specific product/service.",
        "parameters": {
          "type": "object",
          "properties": {
            "product_id": {
              "type": "string",
              "description": "ID of the product/service."
            },
            "quantity": {
              "type": "integer",
              "description": "Estimated quantity required."
            },
            "vendors": {
              "type": "array",
              "items": {
                "type": "object",
                "properties": {
                  "vendor_id": {
                    "type": "string"
                  },
                  "priority": {
                    "type": "integer",
                    "description": "Vendor priority (1=highest, 5=lowest)."
                  }
                },
                "required": [
                  "vendor_id"
                ]
              }
            },
            "deadline": {
              "type": "string",
              "format": "date",
              "description": "Quotation submission deadline (YYYY-MM-DD)."
            },
            "custom_requirements": {
              "type": "string",
              "description": "Additional terms/conditions for the quotation."
            }
          },
          "required": [
            "product_id",
            "quantity",
            "vendors",
            "deadline"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "approve_procurement_request",
        "description": "Approve/reject a procurement request with optional comments.",
        "parameters": {
          "type": "object",
          "properties": {
            "request_id": {
              "type": "string",
              "description": "ID of the procurement request."
            },
            "approver_id": {
              "type": "string",
              "description": "ID of the approving manager."
            },
            "status": {
              "type": "string",
              "enum": [
                "Approved",
                "Rejected",
                "Pending"
              ],
              "description": "Approval status."
            },
            "comments": {
              "type": "string",
              "description": "Optional comments for approval/rejection."
            },
            "budget_check": {
              "type": "boolean",
              "description": "Whether to enforce budget validation (default: true).",
              "default": True
            }
          },
          "required": [
            "request_id",
            "approver_id",
            "status"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "create_sales_proposal",
        "description": "Generate a sales proposal for a client.",
        "parameters": {
          "type": "object",
          "properties": {
            "client_id": {
              "type": "string",
              "description": "Client ID."
            },
            "proposal_title": {
              "type": "string",
              "description": "Proposal title."
            },
            "products": {
              "type": "array",
              "description": "Products/services in proposal.",
              "items": {
                "type": "object",
                "properties": {
                  "product_id": {
                    "type": "string",
                    "description": "Product ID."
                  },
                  "quantity": {
                    "type": "integer",
                    "description": "Quantity."
                  },
                  "unit_price": {
                    "type": "number",
                    "description": "Price per unit."
                  }
                },
                "required": [
                  "product_id",
                  "quantity",
                  "unit_price"
                ]
              }
            },
            "payment_terms": {
              "type": "object",
              "description": "Payment terms.",
              "properties": {
                "due_in_days": {
                  "type": "integer",
                  "description": "Days until payment due."
                }
              }
            }
          },
          "required": [
            "client_id",
            "proposal_title",
            "products"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "generate_sales_report",
        "description": "Generate sales performance reports.",
        "parameters": {
          "type": "object",
          "properties": {
            "report_type": {
              "type": "string",
              "enum": [
                "individual_performance",
                "team_performance",
                "pipeline_analysis",
                "win_loss_analysis",
                "forecast_accuracy"
              ],
              "description": "Type of report"
            },
            "date_range": {
              "type": "object",
              "properties": {
                "start_date": {
                  "type": "string",
                  "description": "Start date (YYYY-MM-DD)"
                },
                "end_date": {
                  "type": "string",
                  "description": "End date (YYYY-MM-DD)"
                }
              },
              "required": [
                "start_date",
                "end_date"
              ]
            },
            "filters": {
              "type": "object",
              "properties": {
                "sales_rep_ids": {
                  "type": "array",
                  "items": {
                    "type": "string"
                  },
                  "description": "Sales reps to filter"
                },
                "product_categories": {
                  "type": "array",
                  "items": {
                    "type": "string"
                  },
                  "description": "Product categories to filter"
                }
              }
            },
            "metrics_to_include": {
              "type": "array",
              "items": {
                "type": "string",
                "enum": [
                  "revenue",
                  "deals_closed",
                  "conversion_rate",
                  "average_deal_size",
                  "sales_cycle_length",
                  "quota_attainment"
                ]
              },
              "description": "Metrics to include"
            },
            "output_format": {
              "type": "string",
              "enum": [
                "pdf",
                "excel",
                "dashboard",
                "email"
              ],
              "description": "Output format"
            }
          },
          "required": [
            "report_type",
            "date_range",
            "metrics_to_include"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "generate_sales_forecast",
        "description": "Generate sales forecast report.",
        "parameters": {
          "type": "object",
          "properties": {
            "forecast_period": {
              "type": "object",
              "description": "Forecast time range.",
              "properties": {
                "start_date": {
                  "type": "string",
                  "description": "Start date (YYYY-MM-DD)."
                },
                "end_date": {
                  "type": "string",
                  "description": "End date (YYYY-MM-DD)."
                },
                "forecast_type": {
                  "type": "string",
                  "enum": [
                    "monthly",
                    "quarterly",
                    "annual"
                  ]
                }
              },
              "required": [
                "start_date",
                "end_date"
              ]
            },
            "territories": {
              "type": "array",
              "description": "Territories to include.",
              "items": {
                "type": "string"
              }
            },
            "product_categories": {
              "type": "array",
              "description": "Product categories to analyze.",
              "items": {
                "type": "string"
              }
            },
            "confidence_intervals": {
              "type": "array",
              "description": "Confidence levels (e.g., 0.8, 0.9).",
              "items": {
                "type": "number"
              }
            }
          },
          "required": [
            "forecast_period",
            "territories",
            "product_categories"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "conduct_compliance_audit",
        "description": "Execute a compliance audit with risk assessment and reporting.",
        "parameters": {
          "type": "object",
          "properties": {
            "audit_id": {
              "type": "string",
              "description": "Unique identifier for the audit"
            },
            "regulatory_frameworks": {
              "type": "array",
              "items": {
                "type": "string",
                "enum": ["GDPR", "HIPAA", "SOX", "PCI_DSS", "ISO_27001", "CCPA", "SOC2", "FISMA"]
              },
              "description": "Frameworks to audit against"
            },
            "business_units": {
              "type": "array",
              "items": {
                "type": "string"
              },
              "description": "Business units to include"
            },
            "audit_period": {
              "type": "object",
              "properties": {
                "start_date": {
                  "type": "string",
                  "description": "Audit start date (YYYY-MM-DD)"
                },
                "end_date": {
                  "type": "string",
                  "description": "Audit end date (YYYY-MM-DD)"
                }
              },
              "required": ["start_date", "end_date"]
            },
            "lead_auditor": {
              "type": "string",
              "description": "Employee ID of lead auditor"
            },
            "risk_categories": {
              "type": "array",
              "items": {
                "type": "string",
                "enum": ["data_privacy", "financial_reporting", "operational", "security", "environmental", "employment"]
              },
              "description": "Risk categories to evaluate"
            },
            "materiality_threshold": {
              "type": "number",
              "description": "Financial materiality threshold"
            },
            "automated_testing": {
              "type": "boolean",
              "default": True,
              "description": "Enable automated testing"
            }
          },
          "required": ["audit_id", "regulatory_frameworks", "business_units", "audit_period", "lead_auditor", "risk_categories"]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "track_regulatory_changes",
        "description": "Monitor regulatory changes and assess compliance impact.",
        "parameters": {
          "type": "object",
          "properties": {
            "jurisdictions": {
              "type": "array",
              "description": "Legal jurisdictions to monitor (e.g., ['EU', 'US'])",
              "items": {
                "type": "string"
              }
            },
            "regulatory_areas": {
              "type": "array",
              "description": "Areas to track (e.g., ['data_protection', 'tax'])",
              "items": {
                "type": "string"
              }
            },
            "sources": {
              "type": "array",
              "description": "Monitoring sources (e.g., government websites)",
              "items": {
                "type": "object",
                "properties": {
                  "type": {
                    "type": "string",
                    "enum": [
                      "government_website",
                      "legal_database"
                    ]
                  },
                  "name": {
                    "type": "string",
                    "description": "Source name"
                  }
                },
                "required": [
                  "type",
                  "name"
                ]
              }
            },
            "notify_groups": {
              "type": "array",
              "description": "Stakeholders to notify (e.g., ['legal@company.com'])",
              "items": {
                "type": "string"
              }
            }
          },
          "required": [
            "jurisdictions",
            "regulatory_areas",
            "sources",
            "notify_groups"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "escalate_policy_violation",
        "description": "Escalate a policy violation to the legal team.",
        "parameters": {
          "type": "object",
          "properties": {
            "violation_id": {
              "type": "string",
              "description": "Unique ID of the violation"
            },
            "description": {
              "type": "string",
              "description": "Brief summary of the violation"
            },
            "severity": {
              "type": "string",
              "enum": [
                "minor",
                "moderate",
                "critical"
              ],
              "description": "Severity level"
            },
            "evidence": {
              "type": "array",
              "items": {
                "type": "object",
                "properties": {
                  "type": {
                    "type": "string",
                    "enum": [
                      "log",
                      "screenshot",
                      "email"
                    ]
                  },
                  "content": {
                    "type": "string"
                  }
                },
                "required": [
                  "type",
                  "content"
                ]
              },
              "description": "Supporting evidence"
            },
            "notify_legal_team": {
              "type": "boolean",
              "default": True,
              "description": "Alert legal team immediately"
            },
            "auto_lock_account": {
              "type": "boolean",
              "description": "Lock the violator's account if critical.",
              "default": False
            }
          },
          "required": [
            "violation_id",
            "description",
            "severity",
            "evidence"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "submit_maintenance_request",
        "description": "Submit a maintenance request for facility issues.",
        "parameters": {
          "type": "object",
          "properties": {
            "location": {
              "type": "string",
              "description": "Specific location of the issue (e.g., 'Room 301', 'Parking Lot B')"
            },
            "issue_type": {
              "type": "string",
              "enum": [
                "plumbing",
                "electrical",
                "hvac",
                "cleaning",
                "security",
                "equipment",
                "other"
              ],
              "description": "Category of the maintenance issue"
            },
            "priority": {
              "type": "string",
              "enum": [
                "low",
                "medium",
                "high",
                "emergency"
              ],
              "description": "Priority level"
            },
            "description": {
              "type": "string",
              "description": "Brief description of the issue"
            },
            "requester_id": {
              "type": "string",
              "description": "Employee ID of the requester"
            }
          },
          "required": [
            "location",
            "issue_type",
            "description",
            "requester_id"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "book_meeting_room",
        "description": "Reserve a meeting room for a specific date and time.",
        "parameters": {
          "type": "object",
          "properties": {
            "room_id": {
              "type": "string",
              "description": "Unique meeting room ID (e.g., 'CONF-A-301')"
            },
            "date": {
              "type": "string",
              "description": "Booking date (YYYY-MM-DD)"
            },
            "start_time": {
              "type": "string",
              "description": "Start time (HH:MM, 24-hour)"
            },
            "end_time": {
              "type": "string",
              "description": "End time (HH:MM, 24-hour)"
            },
            "attendee_count": {
              "type": "integer",
              "description": "Number of attendees (1-100)",
              "minimum": 1,
              "maximum": 100
            },
            "meeting_title": {
              "type": "string",
              "description": "Title of the meeting"
            },
            "organizer_id": {
              "type": "string",
              "description": "Employee ID of the organizer"
            },
            "equipment_needed": {
              "type": "array",
              "items": {
                "type": "string"
              },
              "description": "Equipment required (e.g., 'projector')"
            }
          },
          "required": [
            "room_id",
            "date",
            "start_time",
            "end_time",
            "attendee_count",
            "meeting_title",
            "organizer_id"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "monitor_energy_consumption",
        "description": "Track and analyze energy usage across facility zones.",
        "parameters": {
          "type": "object",
          "properties": {
            "building_id": {
              "type": "string",
              "description": "ID of the building to monitor."
            },
            "time_range": {
              "type": "object",
              "description": "Time period for energy analysis (start/end in ISO format).",
              "properties": {
                "start": {
                  "type": "string"
                },
                "end": {
                  "type": "string"
                }
              },
              "required": [
                "start",
                "end"
              ]
            },
            "granularity": {
              "type": "string",
              "enum": [
                "hourly",
                "daily",
                "weekly",
                "monthly"
              ],
              "description": "Time granularity for reports."
            },
            "threshold_kwh": {
              "type": "number",
              "description": "Alert if energy usage exceeds this value (kWh)."
            }
          },
          "required": [
            "building_id",
            "time_range"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "track_shipment",
        "description": "Track the current status and location of a shipment using tracking number.",
        "parameters": {
          "type": "object",
          "properties": {
            "tracking_number": {
              "type": "string",
              "description": "The unique tracking identifier for the shipment"
            },
            "carrier": {
              "type": "string",
              "description": "Shipping carrier name",
              "enum": [
                "FedEx",
                "UPS",
                "DHL",
                "USPS",
                "Amazon Logistics"
              ]
            },
            "include_history": {
              "type": "boolean",
              "description": "Whether to include full tracking history",
              "default": False
            }
          },
          "required": [
            "tracking_number",
            "carrier"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "create_shipping_label",
        "description": "Generate a shipping label for package delivery.",
        "parameters": {
          "type": "object",
          "properties": {
            "sender": {
              "type": "object",
              "properties": {
                "name": {
                  "type": "string"
                },
                "address": {
                  "type": "string"
                },
                "city": {
                  "type": "string"
                },
                "state": {
                  "type": "string"
                },
                "zip_code": {
                  "type": "string"
                },
                "country": {
                  "type": "string"
                }
              },
              "required": [
                "name",
                "address",
                "city",
                "state",
                "zip_code",
                "country"
              ]
            },
            "recipient": {
              "type": "object",
              "properties": {
                "name": {
                  "type": "string"
                },
                "address": {
                  "type": "string"
                },
                "city": {
                  "type": "string"
                },
                "state": {
                  "type": "string"
                },
                "zip_code": {
                  "type": "string"
                },
                "country": {
                  "type": "string"
                }
              },
              "required": [
                "name",
                "address",
                "city",
                "state",
                "zip_code",
                "country"
              ]
            },
            "package": {
              "type": "object",
              "properties": {
                "weight": {
                  "type": "number",
                  "description": "Weight in pounds"
                },
                "length": {
                  "type": "number",
                  "description": "Length in inches"
                },
                "width": {
                  "type": "number",
                  "description": "Width in inches"
                },
                "height": {
                  "type": "number",
                  "description": "Height in inches"
                }
              },
              "required": [
                "weight",
                "length",
                "width",
                "height"
              ]
            },
            "service_type": {
              "type": "string",
              "enum": [
                "standard",
                "express",
                "overnight",
                "two_day"
              ],
              "description": "Shipping speed"
            }
          },
          "required": [
            "sender",
            "recipient",
            "package"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "calculate_shipping_cost",
        "description": "Estimate shipping costs based on package details, route, and carrier options.",
        "parameters": {
          "type": "object",
          "properties": {
            "origin_postal_code": {
              "type": "string",
              "description": "Origin postal code."
            },
            "destination_postal_code": {
              "type": "string",
              "description": "Destination postal code."
            },
            "package_details": {
              "type": "array",
              "items": {
                "type": "object",
                "properties": {
                  "weight_kg": {
                    "type": "number",
                    "description": "Weight in kilograms."
                  },
                  "dimensions_cm": {
                    "type": "object",
                    "properties": {
                      "length": {
                        "type": "number"
                      },
                      "width": {
                        "type": "number"
                      },
                      "height": {
                        "type": "number"
                      }
                    },
                    "required": [
                      "length",
                      "width",
                      "height"
                    ]
                  }
                },
                "required": [
                  "weight_kg",
                  "dimensions_cm"
                ]
              }
            },
            "carrier_options": {
              "type": "array",
              "items": {
                "type": "string",
                "enum": [
                  "FedEx",
                  "UPS",
                  "DHL",
                  "USPS"
                ]
              }
            },
            "service_level": {
              "type": "string",
              "enum": [
                "standard",
                "express",
                "overnight"
              ]
            }
          },
          "required": [
            "origin_postal_code",
            "destination_postal_code",
            "package_details"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "create_support_ticket",
        "description": "Create a customer support ticket.",
        "parameters": {
          "type": "object",
          "properties": {
            "customer_id": {
              "type": "string",
              "description": "Unique customer identifier."
            },
            "issue_category": {
              "type": "string",
              "enum": [
                "technical",
                "billing",
                "product_inquiry",
                "complaint",
                "feature_request",
                "account_access"
              ],
              "description": "Category of the issue."
            },
            "priority": {
              "type": "string",
              "enum": [
                "low",
                "medium",
                "high",
                "critical"
              ],
              "description": "Ticket priority."
            },
            "subject": {
              "type": "string",
              "description": "Brief issue summary."
            },
            "description": {
              "type": "string",
              "description": "Detailed issue description."
            },
            "product_name": {
              "type": "string",
              "description": "Name of the product related to the issue."
            },
            "customer_contact": {
              "type": "object",
              "properties": {
                "email": {
                  "type": "string",
                  "description": "Customer email (required)."
                }
              },
              "required": [
                "email"
              ]
            },
            "department": {
              "type": "string",
              "enum": [
                "technical_support",
                "billing",
                "sales",
                "general"
              ],
              "description": "Department to route to."
            }
          },
          "required": [
            "customer_id",
            "issue_category",
            "subject",
            "description",
            "product_name",
            "customer_contact"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "escalate_support_ticket",
        "description": "Escalate a support ticket to a higher-priority team.",
        "parameters": {
          "type": "object",
          "properties": {
            "ticket_id": {
              "type": "string",
              "description": "Support ticket ID."
            },
            "reason": {
              "type": "string",
              "description": "Reason for escalation (e.g., 'critical', 'unresolved')."
            },
            "target_team": {
              "type": "string",
              "enum": [
                "senior_support",
                "engineering",
                "billing"
              ],
              "description": "Team to escalate to."
            },
            "notify_customer": {
              "type": "boolean",
              "default": True,
              "description": "Notify customer about escalation."
            }
          },
          "required": [
            "ticket_id",
            "reason",
            "target_team"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "initiate_live_chat_session",
        "description": "Start a live chat session between customer and agent.",
        "parameters": {
          "type": "object",
          "properties": {
            "customer_id": {
              "type": "string",
              "description": "Unique ID of the customer."
            },
            "chat_type": {
              "type": "string",
              "enum": [
                "sales_inquiry",
                "technical_support",
                "billing_question",
                "general_support"
              ],
              "description": "Type of chat requested."
            },
            "customer_context": {
              "type": "object",
              "properties": {
                "account_tier": {
                  "type": "string",
                  "enum": [
                    "free",
                    "basic",
                    "premium",
                    "enterprise"
                  ],
                  "description": "Customer's subscription tier."
                },
                "previous_tickets": {
                  "type": "array",
                  "items": {
                    "type": "string"
                  },
                  "description": "List of recent ticket IDs for context"
                },
                "current_page": {
                  "type": "string",
                  "description": "Page customer is viewing."
                }
              },
              "required": [
                "account_tier"
              ]
            },
            "queue_preferences": {
              "type": "object",
              "properties": {
                "preferred_language": {
                  "type": "string",
                  "description": "Preferred language code (e.g., 'en')."
                },
                "max_wait_time": {
                  "type": "integer",
                  "description": "Max wait time (seconds) before callback."
                }
              }
            },
            "urgency_level": {
              "type": "string",
              "enum": [
                "low",
                "normal",
                "high",
                "emergency"
              ],
              "description": "Urgency of the inquiry."
            }
          },
          "required": [
            "customer_id",
            "chat_type",
            "customer_context"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "create_performance_review",
        "description": "Create a performance review for an employee.",
        "parameters": {
          "type": "object",
          "properties": {
            "employee_id": {
              "type": "string",
              "description": "ID of the employee being reviewed"
            },
            "reviewer_id": {
              "type": "string",
              "description": "ID of the reviewer (manager/supervisor)"
            },
            "review_period": {
              "type": "object",
              "properties": {
                "start_date": {
                  "type": "string",
                  "description": "Review period start date (YYYY-MM-DD)"
                },
                "end_date": {
                  "type": "string",
                  "description": "Review period end date (YYYY-MM-DD)"
                }
              },
              "required": [
                "start_date",
                "end_date"
              ]
            },
            "overall_rating": {
              "type": "integer",
              "description": "Performance rating (1-5 scale)",
              "minimum": 1,
              "maximum": 5
            },
            "review_type": {
              "type": "string",
              "enum": [
                "annual",
                "mid_year",
                "quarterly",
                "probationary"
              ],
              "description": "Type of review"
            }
          },
          "required": [
            "employee_id",
            "reviewer_id",
            "review_period",
            "overall_rating"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "schedule_performance_review_meeting",
        "description": "Schedule a performance review meeting.",
        "parameters": {
          "type": "object",
          "properties": {
            "employee_id": {
              "type": "string",
              "description": "Employee ID"
            },
            "manager_id": {
              "type": "string",
              "description": "Manager ID"
            },
            "meeting_details": {
              "type": "object",
              "properties": {
                "proposed_datetime": {
                  "type": "string",
                  "description": "Meeting datetime (ISO 8601)"
                },
                "duration_minutes": {
                  "type": "integer",
                  "description": "Meeting duration (minutes)"
                },
                "location": {
                  "type": "string",
                  "description": "Room/link for meeting"
                }
              },
              "required": [
                "proposed_datetime",
                "location"
              ]
            },
            "review_type": {
              "type": "string",
              "enum": [
                "annual",
                "mid_year",
                "quarterly",
                "probationary"
              ],
              "description": "Type of review"
            },
            "preparation_required": {
              "type": "boolean",
              "default": True,
              "description": "Requires pre-meeting prep"
            },
            "send_calendar_invite": {
              "type": "boolean",
              "default": True,
              "description": "Auto-send calendar invite"
            }
          },
          "required": [
            "employee_id",
            "manager_id",
            "meeting_details",
            "review_type"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "request_peer_feedback",
        "description": "Request peer feedback for an employee.",
        "parameters": {
          "type": "object",
          "properties": {
            "employee_id": {
              "type": "string",
              "description": "Employee ID for feedback."
            },
            "reviewer_ids": {
              "type": "array",
              "items": {
                "type": "string",
                "description": "Peer reviewer IDs."
              },
              "minItems": 1,
              "description": "At least one peer required."
            },
            "deadline": {
              "type": "string",
              "description": "Feedback deadline (YYYY-MM-DD)."
            },
            "feedback_criteria": {
              "type": "array",
              "items": {
                "type": "string",
                "enum": [
                  "Teamwork",
                  "Problem-Solving",
                  "Leadership",
                  "Creativity"
                ]
              },
              "description": "Feedback focus areas."
            },
            "anonymous": {
              "type": "boolean",
              "default": True,
              "description": "Anonymous feedback toggle."
            }
          },
          "required": [
            "employee_id",
            "reviewer_ids",
            "deadline"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "schedule_meeting",
        "description": "Schedule a meeting with participants and basic details.",
        "parameters": {
          "type": "object",
          "properties": {
            "title": {
              "type": "string",
              "description": "Meeting title."
            },
            "start_time": {
              "type": "string",
              "description": "Start time (ISO format)."
            },
            "end_time": {
              "type": "string",
              "description": "End time (ISO format)."
            },
            "participants": {
              "type": "array",
              "items": {
                "type": "object",
                "properties": {
                  "email": {
                    "type": "string",
                    "description": "Participant's email address."
                  },
                  "role": {
                    "type": "string",
                    "enum": [
                      "required",
                      "optional"
                    ]
                  }
                },
                "required": [
                  "email"
                ]
              }
            },
            "agenda": {
              "type": "string",
              "description": "Brief agenda (optional)."
            },
            "video_call_link": {
              "type": "string",
              "description": "URL for virtual meeting (optional)."
            }
          },
          "required": [
            "title",
            "start_time",
            "end_time",
            "participants"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "reschedule_meeting",
        "description": "Reschedule an existing meeting with new time or participants.",
        "parameters": {
          "type": "object",
          "properties": {
            "meeting_id": {
              "type": "string",
              "description": "Unique ID of the meeting to reschedule."
            },
            "new_start_time": {
              "type": "string",
              "description": "New start time in ISO 8601 format."
            },
            "new_end_time": {
              "type": "string",
              "description": "New end time in ISO 8601 format."
            },
            "notify_participants": {
              "type": "boolean",
              "default": True
            },
            "additional_participants": {
              "type": "array",
              "items": {
                "type": "string",
                "description": "Emails of new participants to add."
              }
            },
            "reason": {
              "type": "string",
              "description": "Reason for rescheduling."
            }
          },
          "required": [
            "meeting_id",
            "new_start_time",
            "new_end_time"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "cancel_meeting",
        "description": "Cancel a scheduled meeting and notify participants.",
        "parameters": {
          "type": "object",
          "properties": {
            "meeting_id": {
              "type": "string",
              "description": "Unique ID of the meeting to cancel."
            },
            "notify_participants": {
              "type": "boolean",
              "default": True
            },
            "reason": {
              "type": "string",
              "description": "Reason for cancellation."
            },
            "send_alternative_proposal": {
              "type": "boolean",
              "default": False
            },
            "alternative_times": {
              "type": "array",
              "items": {
                "type": "string",
                "description": "Proposed alternative times in ISO 8601 format."
              }
            }
          },
          "required": [
            "meeting_id",
            "reason"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "create_campaign",
        "description": "Create a marketing campaign with core targeting, budget, and creatives.",
        "parameters": {
          "type": "object",
          "properties": {
            "campaign_name": {
              "type": "string",
              "description": "Name of the campaign"
            },
            "campaign_type": {
              "type": "string",
              "enum": [
                "brand_awareness",
                "lead_generation",
                "conversion",
                "retention",
                "product_launch"
              ],
              "description": "Primary goal of the campaign"
            },
            "target_audience": {
              "type": "object",
              "properties": {
                "age_range": {
                  "type": "object",
                  "properties": {
                    "min_age": {
                      "type": "integer",
                      "minimum": 13
                    },
                    "max_age": {
                      "type": "integer",
                      "maximum": 100
                    }
                  },
                  "required": [
                    "min_age",
                    "max_age"
                  ]
                },
                "countries": {
                  "type": "array",
                  "items": {
                    "type": "string"
                  },
                  "description": "Target countries (e.g., ['US', 'UK'])"
                },
                "interests": {
                  "type": "array",
                  "items": {
                    "type": "string"
                  },
                  "description": "Key interest tags (e.g., ['fitness', 'tech'])"
                }
              },
              "required": [
                "age_range",
                "countries"
              ]
            },
            "budget": {
              "type": "object",
              "properties": {
                "total_budget": {
                  "type": "number",
                  "minimum": 100,
                  "description": "Total budget in USD"
                },
                "daily_budget": {
                  "type": "number",
                  "minimum": 10,
                  "description": "Daily spend limit"
                },
                "platforms": {
                  "type": "object",
                  "properties": {
                    "facebook_ads": {
                      "type": "number",
                      "minimum": 0,
                      "maximum": 1
                    },
                    "google_ads": {
                      "type": "number",
                      "minimum": 0,
                      "maximum": 1
                    },
                    "instagram_ads": {
                      "type": "number",
                      "minimum": 0,
                      "maximum": 1
                    }
                  },
                  "required": [
                    "facebook_ads",
                    "google_ads"
                  ]
                }
              },
              "required": [
                "total_budget",
                "platforms"
              ]
            },
            "creative_assets": {
              "type": "object",
              "properties": {
                "primary_message": {
                  "type": "string",
                  "description": "Main campaign text"
                },
                "call_to_action": {
                  "type": "string",
                  "enum": [
                    "learn_more",
                    "shop_now",
                    "sign_up",
                    "download"
                  ]
                },
                "image_url": {
                  "type": "string",
                  "format": "uri",
                  "description": "Primary image URL"
                }
              },
              "required": [
                "primary_message",
                "call_to_action"
              ]
            },
            "schedule": {
              "type": "object",
              "properties": {
                "start_date": {
                  "type": "string",
                  "format": "date",
                  "description": "Campaign start date (YYYY-MM-DD)"
                },
                "end_date": {
                  "type": "string",
                  "format": "date",
                  "description": "Campaign end date (YYYY-MM-DD)"
                }
              },
              "required": [
                "start_date",
                "end_date"
              ]
            }
          },
          "required": [
            "campaign_name",
            "campaign_type",
            "target_audience",
            "budget",
            "creative_assets",
            "schedule"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "analyze_campaign_performance",
        "description": "Analyze marketing campaign metrics across platforms.",
        "parameters": {
          "type": "object",
          "properties": {
            "campaign_ids": {
              "type": "array",
              "items": {
                "type": "string"
              },
              "description": "List of campaign IDs to analyze."
            },
            "date_range": {
              "type": "object",
              "properties": {
                "start_date": {
                  "type": "string",
                  "format": "date",
                  "description": "Start date (YYYY-MM-DD)."
                },
                "end_date": {
                  "type": "string",
                  "format": "date",
                  "description": "End date (YYYY-MM-DD)."
                }
              },
              "required": [
                "start_date",
                "end_date"
              ],
              "description": "Date range for analysis."
            },
            "metrics": {
              "type": "array",
              "items": {
                "type": "string"
              },
              "description": "Metrics to analyze (e.g., clicks, impressions, conversions)."
            },
            "platforms": {
              "type": "array",
              "items": {
                "type": "string"
              },
              "description": "Platforms to include (e.g., facebook, google)."
            },
            "report_format": {
              "type": "string",
              "enum": [
                "pdf",
                "excel",
                "csv",
                "json"
              ],
              "description": "Format of the generated report."
            }
          },
          "required": [
            "campaign_ids",
            "date_range",
            "metrics",
            "platforms"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "track_ad_performance",
        "description": "Monitor real-time performance metrics for paid ads across platforms.",
        "parameters": {
          "type": "object",
          "properties": {
            "ad_ids": {
              "type": "array",
              "items": {
                "type": "string"
              },
              "description": "List of ad IDs to track."
            },
            "platform": {
              "type": "string",
              "enum": [
                "google_ads",
                "facebook_ads",
                "linkedin_ads"
              ],
              "description": "Advertising platform."
            },
            "metrics": {
              "type": "array",
              "items": {
                "type": "string",
                "enum": [
                  "impressions",
                  "clicks",
                  "cost_per_click",
                  "conversion_rate"
                ]
              },
              "description": "Metrics to fetch."
            },
            "polling_interval_minutes": {
              "type": "integer",
              "description": "How often to refresh data."
            }
          },
          "required": [
            "ad_ids",
            "platform",
            "metrics"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "post_job_opening",
        "description": "Post a job opening to selected job boards.",
        "parameters": {
          "type": "object",
          "properties": {
            "job_title": {
              "type": "string",
              "description": "Job title (e.g., 'Senior Software Engineer')."
            },
            "job_description": {
              "type": "string",
              "description": "Summary of job responsibilities."
            },
            "location": {
              "type": "string",
              "description": "Job location (e.g., 'Remote', 'New York, NY')."
            },
            "salary_min": {
              "type": "number",
              "description": "Minimum salary in USD."
            },
            "salary_max": {
              "type": "number",
              "description": "Maximum salary in USD."
            },
            "required_skills": {
              "type": "array",
              "items": {
                "type": "string"
              },
              "description": "Key skills needed (e.g., ['Python', 'AWS'])."
            },
            "job_type": {
              "type": "string",
              "enum": [
                "Full-time",
                "Part-time",
                "Contract",
                "Internship"
              ],
              "description": "Type of employment."
            },
            "post_to_boards": {
              "type": "array",
              "items": {
                "type": "string"
              },
              "description": "Job boards to post to (e.g., ['LinkedIn', 'Indeed'])."
            }
          },
          "required": [
            "job_title",
            "job_description",
            "location",
            "required_skills"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "schedule_interview",
        "description": "Schedule an interview with a candidate, including calendar invites and reminders.",
        "parameters": {
          "type": "object",
          "properties": {
            "candidate_id": {
              "type": "string",
              "description": "Unique identifier for the candidate."
            },
            "interviewer_ids": {
              "type": "array",
              "items": {
                "type": "string"
              },
              "description": "List of interviewer IDs."
            },
            "interview_type": {
              "type": "string",
              "enum": [
                "Phone",
                "Video",
                "On-site"
              ]
            },
            "scheduled_time": {
              "type": "string",
              "format": "date-time",
              "description": "Scheduled interview time (ISO 8601)."
            },
            "duration_minutes": {
              "type": "integer",
              "description": "Duration of the interview in minutes."
            },
            "meeting_link": {
              "type": "string",
              "description": "URL for video interviews (if applicable)."
            },
            "send_reminder": {
              "type": "boolean",
              "default": True,
              "description": "Send a reminder 24 hours before the interview."
            }
          },
          "required": [
            "candidate_id",
            "interviewer_ids",
            "scheduled_time"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "send_offer_letter",
        "description": "Send a job offer letter to a candidate.",
        "parameters": {
          "type": "object",
          "properties": {
            "candidate_id": {
              "type": "string",
              "description": "Candidate's unique ID."
            },
            "job_title": {
              "type": "string",
              "description": "Job title for the offer."
            },
            "salary_base": {
              "type": "number",
              "description": "Base salary amount."
            },
            "start_date": {
              "type": "string",
              "description": "Start date (YYYY-MM-DD)."
            },
            "expiration_days": {
              "type": "integer",
              "description": "Days before offer expires."
            }
          },
          "required": [
            "candidate_id",
            "job_title",
            "salary_base",
            "start_date"
          ]
        }
      }
    }
]

# Main code

In [25]:
# Print Col Names
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("kunjanshah/new_single_turn_function_calling_split", split="test")  # Replace with actual dataset name
df = pd.DataFrame(dataset)
df.columns.tolist()

c:\Users\KUNJAN SHAH\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['id',
 'Turn Type',
 'Prompt_type',
 'tool_needed',
 'tool_call',
 'tool_name',
 'Query',
 'Ground_truth',
 'Missing_para',
 'tools_in_system_prompt',
 'System_Prompt',
 'Conversation']

In [26]:
%%time
import pandas as pd
import json
import os
import time
from datasets import load_dataset
#from openai import OpenAI
#from secret_key import openai_api_key

# client = OpenAI(api_key=openai_api_key)

def get_function_calls(prompt, tools, model_name):
    try:
        start_time = time.time()
        
        response = client.chat.completions.create(
            model=model_name,
messages = [
            {
            "role":"system",
            "content": 
            """You are an AI assistant designed to follow a strict workflow for tool usage. Your primary goal is to accurately call functions based on user requests.

        **Workflow:**
        1.  **Analyze the User's Query:** Carefully understand the user's intent and the information provided.
        2.  **Decide on Tool Use:**
            *   If the query can be answered without a tool, provide a direct text response.
            *   If the query requires an action or external data, you must select and use a tool.
        3.  **Select Tool and Extract Parameters:**
            *   From the `tools` list, select the single most appropriate tool that matches the user's request.
            *   Identify and extract all required parameters for the selected tool directly from the user's query.
            *   For any parameter that is not mentioned in the query but has a `default` value in the tool's schema, you **must** include it in your tool call.
            *   **Do not** include any optional parameters that are not in the query and do not have a default value.
            *   **Do not** invent, assume, or hallucinate any values for parameters. Stick strictly to the information in the query and the default values in the schema.
        4.  **Format the Output:**
            *   If a tool is used, your final output must be a single JSON object representing the tool call.
            """   
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

            tools=tools,
            tool_choice="auto",
            temperature=0.0, 
            parallel_tool_calls=False
        )
        
        execution_time = round(time.time() - start_time, 2)
        
        usage = response.usage
        token_usage = {
            "prompt_tokens": usage.prompt_tokens,
            "completion_tokens": usage.completion_tokens,
            "total_tokens": usage.total_tokens,
            "cached_tokens": getattr(usage.prompt_tokens_details, "cached_tokens", 0) if hasattr(usage, "prompt_tokens_details") else 0,
            "execution_time": execution_time
        }
        
        tool_calls = response.choices[0].message.tool_calls
        text_content = response.choices[0].message.content
        
        if tool_calls:
            tool_call = tool_calls[0]  # Get first tool call
            function_call = {
                "name": tool_call.function.name,
                "arguments": json.loads(tool_call.function.arguments)
            }
            return {
                "response_type": "function_call",
                "function_calls": function_call,
                "usage_metrics": token_usage
            }
        else:
            return {
                "response_type": "text",
                "content": text_content,
                "usage_metrics": token_usage
            }
            
    except Exception as e:
        return {
            "response_type": "error",
            "error": str(e)
        }
# Define the model name
model_name = "qwen/qwen3-14b"
print(f"Using model: {model_name}")

# # Load dataset from Hugging Face
# dataset = load_dataset("kunjanshah/new_single_turn_function_calling_split", split="test")  # Replace with actual dataset name
# df = pd.DataFrame(dataset)

try:
    # Try reading the Excel file
    df = pd.read_excel("D:\\Uni Study\\Sem-6\\6.My Code\\1.Single_turn\\data\\function_calling_test_data.xlsx", engine='openpyxl')
except Exception as e:
    print(f"Error reading Excel file: {e}")
    # Fallback to loading from Hugging Face dataset
    print("Falling back to loading from Hugging Face dataset...")
    dataset = load_dataset("kunjanshah/new_single_turn_function_calling_split", split="test")
    df = pd.DataFrame(dataset)
    # Save to Excel for future use
    os.makedirs("D:\\Uni Study\\Sem-6\\6.My Code\\1.Single_turn\\data", exist_ok=True)
    df.to_excel("D:\\Uni Study\\Sem-6\\6.My Code\\1.Single_turn\\data\\function_calling_test_data.xlsx", index=False)

# Get original column count for reference
original_cols = df.columns.tolist()
print(f"Original dataset has {len(original_cols)} columns: {original_cols}")

# Add new columns
df["Actual_model_call"] = None
df["Token_Usage"] = None
df["Execution_Time"] = None

print(f"Dataset now has {len(df.columns)} columns: {df.columns.tolist()}")

# Process all rows
output_file_all = f"D:\\Uni Study\\Sem-6\\6.My Code\\1.Single_turn\\4.Qwen_single_FC\\qwen_latest_results\\results_{model_name}.xlsx"  # Include model name in filename
# Create directory if it doesn't exist
os.makedirs(os.path.dirname(output_file_all), exist_ok=True)

for idx, row in df.iterrows():
    prompt = str(row["Query"])
    print(f"Processing row {idx+1} with query: {prompt[:50]}...")

    result = get_function_calls(prompt, tools, model_name)
    if result["response_type"] == "function_call":
        df.at[idx, "Actual_model_call"] = json.dumps(result["function_calls"])
    elif result["response_type"] == "text":
        df.at[idx, "Actual_model_call"] = f"{result['content']}"
    else:
        df.at[idx, "Actual_model_call"] = f"{result['error']}"
    
    if "usage_metrics" in result:
        metrics = result["usage_metrics"]
        df.at[idx, "Token_Usage"] = f"Prompt: {metrics['prompt_tokens']}, Completion: {metrics['completion_tokens']}, Total: {metrics['total_tokens']}, Cached: {metrics['cached_tokens']}"
        df.at[idx, "Execution_Time"] = f"{metrics['execution_time']} seconds"

# Save all rows results (all original columns + 3 new columns)
df.to_excel(output_file_all, index=False)
print(f"Done. All rows results saved to {output_file_all}")
print(f"Final Excel file contains {len(df.columns)} columns: {df.columns.tolist()}")

# # Process first 10 rows
# output_file_10 = f"D:\\Uni Study\\Sem-6\\6.My Code\\1.Single_turn\\4.Qwen_single_FC\\qwen_latest_results\\new_results_qwen3-14b.xlsx" 

# # # Create directory if it doesn't exist
# os.makedirs(os.path.dirname(output_file_10), exist_ok=True)

# for idx, row in df.head(10).iterrows():
#     prompt = str(row["Query"])
#     print(f"Processing row {idx+1} with query: {prompt[:50]}...")

#     result = get_function_calls(prompt, tools, model_name)  # Ensure 'tools' is defined

#     if result["response_type"] == "function_call":
#         df.at[idx, "Actual_model_call"] = json.dumps(result["function_calls"])
#     elif result["response_type"] == "text":
#         df.at[idx, "Actual_model_call"] = f"{result['content']}"
#     else:
#         df.at[idx, "Actual_model_call"] = f"{result['error']}"

#     if "usage_metrics" in result:
#         metrics = result["usage_metrics"]
#         df.at[idx, "Token_Usage"] = f"Prompt: {metrics['prompt_tokens']}, Completion: {metrics['completion_tokens']}, Total: {metrics['total_tokens']}, Cached: {metrics['cached_tokens']}"
#         df.at[idx, "Execution_Time"] = f"{metrics['execution_time']} seconds"

# # # Save first 10 rows results (all original columns + 3 new columns)
# df.head(10).to_excel(output_file_10, index=False)
# print(f"Done. First 10 rows results saved to {output_file_10}")
# print(f"Final Excel file contains {len(df.columns)} columns: {df.columns.tolist()}")

Using model: qwen/qwen3-14b
Original dataset has 12 columns: ['id', 'Turn Type', 'Prompt_type', 'tool_needed', 'tool_call', 'tool_name', 'Query', 'Ground_truth', 'Missing_para', 'tools_in_system_prompt', 'System_Prompt', 'Conversation']
Dataset now has 15 columns: ['id', 'Turn Type', 'Prompt_type', 'tool_needed', 'tool_call', 'tool_name', 'Query', 'Ground_truth', 'Missing_para', 'tools_in_system_prompt', 'System_Prompt', 'Conversation', 'Actual_model_call', 'Token_Usage', 'Execution_Time']
Processing row 1 with query: Can you prepare a sales forecast for the US and EM...
Processing row 2 with query: "Create a support ticket for customer ID CUST789 w...
Processing row 3 with query: Can you hit up VEND001 and VEND002 about pricing f...


KeyboardInterrupt: 

# Single Testing

In [23]:
# Testing prompt
messages = [

            {
            "role": "user",
            "content": "We received an invoice from a supplier, and I’d like to process the payment for INV123. The amount is $5,000, and we’ll use a wire transfer. Can you ensure this gets paid by tomorrow?"

        }
]

In [24]:
# Testing a single query
response = client.chat.completions.create(  
  model="qwen/qwen3-14b", 
  messages=messages,
  tools=tools,
  tool_choice="auto",
  temperature=0.0,
  )
print("1", response)
print("2", response.choices[0].message.content)
#print("3", response.choices[0].message.reasoning)
# Print all function names
if response.choices[0].message.tool_calls:
    for tool_call in response.choices[0].message.tool_calls:
        print("Tool Call:", tool_call)
        print("name", tool_call.function.name)
        print("arguments", tool_call.function.arguments)

        # Extract in your desired format
        try:
            import json
            parsed_args = json.loads(tool_call.function.arguments)
            formatted_call = {
                "name": tool_call.function.name,
                "arguments": parsed_args
            }
            print("Formatted Tool Call:", json.dumps(formatted_call, indent=2))
        except json.JSONDecodeError as e:
            print(f"Error parsing arguments: {e}")
else:
    print("No tool calls found")

1 ChatCompletion(id='gen-1757339698-w4Roe2drfVxDqzmbQ8Oj', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_rK0eitwlSxDAF3jmd7Xi9jhT', function=Function(arguments='{"invoice_id": "INV123", "vendor_id": "SUP456", "amount": 5000, "currency": "USD", "payment_method": "Wire"}', name='process_invoice_payment'), type='function', index=0)], reasoning="Okay, let me see. The user wants to process an invoice payment for INV123, which is $5,000 via wire transfer. They need it done by tomorrow.\n\nFirst, I need to check the available functions. The process_invoice_payment function seems relevant. It requires invoice_id, vendor_id, amount, currency, and payment_method. The user provided the invoice ID, amount, and payment method. But they didn't mention the vendor ID. Hmm, maybe I need to ask for that. Wait,